In [ ]:
import pandas as pd
from pathlib import Path

from transformers import pipeline, AutoTokenizer
from iflip.evaluate.metrics import (
    predict_with_sliding_window,
    compute_perplexity,
    compute_semantic_similarity,
)
from transformers import pipeline, AutoTokenizer
from iflip.config import config


MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}



def flip_rate_multiclass(orig_texts, cf_texts):
    clf_name = config.classifier_model
    clf      = pipeline("text-classification", model=clf_name, device=0)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)


    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds   = predict_with_sliding_window(cf_texts,  clf, tok)
    flips      = sum(o != c for o, c in zip(orig_preds, cf_preds))
    return flips / len(orig_texts), orig_preds, cf_preds


results_dir = Path("results_polyjuice")
csv_files   = sorted(results_dir.glob("*.csv"))
summary_rows = []

for file in csv_files:
    print(f"\n now evaluate: {file.name}")
    try:
        df = pd.read_csv(file)
        df.columns = [c.strip().lower() for c in df.columns]
        orig_col   = [c for c in df.columns if "original"      in c][0]
        cf_col     = [c for c in df.columns if "counterfactual" in c][0]

        originals  = df[orig_col].astype(str).tolist()
        counterfs  = df[cf_col]  .astype(str).tolist()

        dataset    = next(ds for ds in MODEL_MAP if ds in file.stem)
        config.task_name        = dataset
        config.classifier_model = MODEL_MAP[dataset]

        fr, _, _  = flip_rate_multiclass(originals, counterfs)
        ss       = compute_semantic_similarity(originals, counterfs)
        ppl       = compute_perplexity(counterfs)

        summary_rows.append({
            "file"         : file.name,
            "dataset"      : dataset,
            "flip_rate"    : round(fr, 3),
            "semantic_similarity": round(ss, 3),
            "perplexity"   : round(ppl, 2),
        })
        print(f"FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")
    
    except Exception as e:
        print(f"Error -> {file.name} (skipped)")
        print("    Error msg:", e)
        continue


pd.DataFrame(summary_rows)

In [ ]:
import pandas as pd
from pathlib import Path
from transformers import pipeline, AutoTokenizer
from iflip.evaluate.metrics import (
    predict_with_sliding_window,
    compute_perplexity,
    compute_semantic_similarity,
)
from transformers import pipeline, AutoTokenizer
from iflip.config import config


MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}



def flip_rate_multiclass(orig_texts, cf_texts):
    clf_name = config.classifier_model
    clf      = pipeline("text-classification", model=clf_name, device=0)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)


    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds   = predict_with_sliding_window(cf_texts,  clf, tok)
    flips      = sum(o != c for o, c in zip(orig_preds, cf_preds))
    return flips / len(orig_texts), orig_preds, cf_preds



results_dir = Path("results_bae")
csv_files   = sorted(results_dir.glob("*.csv"))
summary_rows = []

for file in csv_files:
    print(f"\n now evaluate: {file.name}")
    try:
        df = pd.read_csv(file)
        df.columns = [c.strip().lower() for c in df.columns]
        orig_col   = [c for c in df.columns if "original"      in c][0]
        cf_col     = [c for c in df.columns if "counterfactual" in c][0]

        originals  = df[orig_col].astype(str).tolist()
        counterfs  = df[cf_col]  .astype(str).tolist()

        dataset    = next(ds for ds in MODEL_MAP if ds in file.stem)
        config.task_name        = dataset
        config.classifier_model = MODEL_MAP[dataset]

        fr, _, _  = flip_rate_multiclass(originals, counterfs)
        ss       = compute_semantic_similarity(originals, counterfs)
        ppl       = compute_perplexity(counterfs)

        summary_rows.append({
            "file"         : file.name,
            "dataset"      : dataset,
            "flip_rate"    : round(fr, 3),
            "semantic_similarity": round(ss, 3),
            "perplexity"   : round(ppl, 2),
        })
        print(f"FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")

    except OverflowError as e:
        print(f"OverflowError  ->  {file.name}  ")
        print("    Error msg:", e)
        continue          


pd.DataFrame(summary_rows)

In [ ]:
import re
import pandas as pd
from pathlib import Path
from transformers import pipeline, AutoTokenizer
from iflip.evaluate.metrics import (
    predict_with_sliding_window,
    compute_perplexity,
    compute_semantic_similarity,
)
from transformers import pipeline, AutoTokenizer
from iflip.config import config


MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}

def parse_model_name(file_path: Path) -> str:
    s = str(file_path)

    m = re.search(r"models--([^/]+)--([^/]+)__snapshots__", s)
    if m:
        org, model = m.group(1), m.group(2)
        return f"{org}/{model}"

    name = file_path.name
    m = re.match(r"([^_]+)__([^_]+)_", name)
    if m:
        org, model = m.group(1), m.group(2)
        return f"{org}/{model}"

    m = re.search(r"([A-Za-z0-9_.-]+)__([A-Za-z0-9_.-]+)", s)
    if m:
        org, model = m.group(1), m.group(2)
        return f"{org}/{model}"

    return "unknown"


def flip_rate_multiclass(orig_texts, cf_texts, clf_name):
    clf = pipeline("text-classification", model=clf_name, device=0)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)
    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds = predict_with_sliding_window(cf_texts, clf, tok)

    flips = sum(1 for o, c in zip(orig_preds, cf_preds) if o != c)
    return flips / len(orig_texts), orig_preds, cf_preds

def filter_bad_samples(originals, counterfs, bad_substr="and closing"):
    good_pairs = [
        (o, c)
        for o, c in zip(originals, counterfs)
        if bad_substr not in o and bad_substr not in c
    ]
    if not good_pairs:
        return [], []
    originals, counterfs = zip(*good_pairs)
    return list(originals), list(counterfs)


results_dir = Path("results_cgg")
summary_rows = []

for file in sorted(results_dir.rglob("*.csv")):
    print(f"\n Evaluating: {file.relative_to(results_dir)}")
    try:

        if file.name.lower() == "evaluation_summary.csv":
            print("Detected evaluation_summary.csv, skipping")
            continue

        df = pd.read_csv(file)
        df.columns = [c.strip().lower() for c in df.columns]


        fname = file.name.lower()
        dataset = None
        for ds in MODEL_MAP.keys():
            if ds in fname:
                dataset = ds
                break
        if dataset is None:
            print(f"Could not identify dataset type -> {file.name}, skipping")
            continue

        clf_name = MODEL_MAP[dataset]
        model_id = parse_model_name(file)

        # ---------------- IMDB / AGNews ----------------
        if dataset in ["imdb", "agnews"]:
            originals = df["orig_text"].astype(str).tolist()
            counterfs = df["gen_text"].astype(str).tolist()

            fr, _, _ = flip_rate_multiclass(originals, counterfs, clf_name)
            sim = compute_semantic_similarity(originals, counterfs)
            ppl = compute_perplexity(counterfs)

            summary_rows.append({
                "file": file.name,
                "model": model_id,
                "dataset": dataset,
                "setting": "default",
                "flip_rate": round(fr, 3),
                "semantic_similarity": round(sim, 3),
                "perplexity": round(ppl, 2),
            })
            print(f"{dataset} | MODEL {model_id} | FR {fr:.3f} | SS {sim:.3f} | PPL {ppl:.2f}")

        # ---------------- SNLI ----------------
        elif dataset == "snli":
            orig_concat = (
                "Premise: " + df["orig_premise"].astype(str) +
                " Hypothesis: " + df["orig_hypothesis"].astype(str)
            ).tolist()


            counterfs_prem = (
                "Premise: " + df["gen_premise"].astype(str) +
                " Hypothesis: " + df["orig_hypothesis"].astype(str)
            ).tolist()

            o1, c1 = filter_bad_samples(orig_concat, counterfs_prem)
            if o1:
                fr, _, _ = flip_rate_multiclass(o1, c1, clf_name)
                ss = compute_semantic_similarity(o1, c1)
                ppl = compute_perplexity(c1)

                summary_rows.append({
                    "file": file.name,
                    "model": model_id,
                    "dataset": dataset,
                    "setting": "premise_replaced",
                    "flip_rate": round(fr, 3),
                    "semantic_similarity": round(ss, 3),
                    "perplexity": round(ppl, 2),
                })
                print(f"SNLI-premise_replaced | MODEL {model_id} | FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")
            else:
                print("No valid SNLI-premise samples after filtering, skipped.")

            # --- Replace hypothesis ---
            counterfs_hypo = (
                "Premise: " + df["orig_premise"].astype(str) +
                " Hypothesis: " + df["gen_hypothesis"].astype(str)
            ).tolist()

            o2, c2 = filter_bad_samples(orig_concat, counterfs_hypo)
            if o2:
                fr, _, _ = flip_rate_multiclass(o2, c2, clf_name)
                ss = compute_semantic_similarity(o2, c2)
                ppl = compute_perplexity(c2)

                summary_rows.append({
                    "file": file.name,
                    "model": model_id,
                    "dataset": dataset,
                    "setting": "hypothesis_replaced",
                    "flip_rate": round(fr, 3),
                    "semantic_similarity": round(ss, 3),
                    "perplexity": round(ppl, 2),
                })
                print(f"SNLI-hypothesis_replaced | MODEL {model_id} | FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")
            else:
                print("No valid SNLI-hypothesis samples after filtering, skipped.")

    except Exception as e:
        print(f"Error -> {file.name} (skipped)")
        print("    Error msg:", e)
        continue


summary_df = pd.DataFrame(summary_rows)

import IPython.display as disp
disp.display(summary_df)

summary_df.to_csv(results_dir / "evaluation_summary.csv", index=False)
print("Summary saved to evaluation_summary.csv")

In [ ]:
import re
import pandas as pd
from pathlib import Path
from transformers import pipeline, AutoTokenizer
import pandas as pd
from pathlib import Path
from iflip.evaluate.metrics import (
    predict_with_sliding_window,
    compute_perplexity,
    compute_semantic_similarity,
)
from iflip.evaluate.metrics import _majority_vote
from transformers import pipeline, AutoTokenizer
from iflip.config import config


MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}


def parse_model_name(file_path: Path) -> str:
    s = str(file_path)

    m = re.search(r"models--([^/]+)--([^/]+)__snapshots__", s)
    if m:
        org, model = m.group(1), m.group(2)
        return f"{org}/{model}"

    name = file_path.name
    m = re.match(r"([^_]+)__([^_]+)_", name)
    if m:
        org, model = m.group(1), m.group(2)
        return f"{org}/{model}"

    m = re.search(r"([A-Za-z0-9_.-]+)__([A-Za-z0-9_.-]+)", s)
    if m:
        org, model = m.group(1), m.group(2)
        return f"{org}/{model}"

    return "unknown"


def flip_rate_multiclass(orig_texts, cf_texts, clf_name):
    clf = pipeline("text-classification", model=clf_name, device=0)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)
    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds = predict_with_sliding_window(cf_texts, clf, tok)

    flips = sum(1 for o, c in zip(orig_preds, cf_preds) if o != c)
    return flips / len(orig_texts), orig_preds, cf_preds


def filter_bad_samples(originals, counterfs, bad_substr="and closing"):
    good_pairs = [
        (o, c)
        for o, c in zip(originals, counterfs)
        if bad_substr not in o and bad_substr not in c
    ]
    if not good_pairs:
        return [], []
    originals, counterfs = zip(*good_pairs)
    return list(originals), list(counterfs)

def get_first_existing_col(df: pd.DataFrame, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

results_dir = Path("results_fizle")
summary_rows = []


all_csvs = [
    p for p in sorted(results_dir.rglob("*.csv"))
    if "data-files" not in p.parts
]

for file in all_csvs:
    print(f"\n Evaluating: {file.relative_to(results_dir)}")
    try:

        if file.name.lower() == "evaluation_summary.csv":
            print("Detected evaluation_summary.csv, skipping")
            continue

        df = pd.read_csv(file)
        df.columns = [c.strip() for c in df.columns]
        df.columns = [c.lower() for c in df.columns]


        fname = file.name.lower()
        dataset = None
        if "imdb" in fname:
            dataset = "imdb"
        elif "ag_news" in fname or "agnews" in fname or "ag-news" in fname:
            dataset = "agnews"
        elif "snli" in fname:
            dataset = "snli"
        else:
            
            if {"premise", "hypothesis"}.issubset(set(df.columns)):
                dataset = "snli"
            elif "original_text" in df.columns and "counterfactual_text" in df.columns:
                
                if "imdb" in str(file).lower():
                    dataset = "imdb"
                elif any(x in str(file).lower() for x in ["ag_news", "agnews", "ag-news"]):
                    dataset = "agnews"

        if dataset is None:
            print(f"Could not identify dataset type -> {file.name}, skipping")
            continue

        clf_name = MODEL_MAP[dataset]
        model_id = parse_model_name(file)

        # ---------------- IMDB / AGNews ----------------
        if dataset in ["imdb", "agnews"]:

            orig_col = get_first_existing_col(df, ["original_text", "orig_text", "text", "input_text"])
            cf_col   = get_first_existing_col(df, ["counterfactual_text", "gen_text", "cf_text"])

            if orig_col is None or cf_col is None:
                print(f"Missing required columns for {dataset} -> {file.name}, skipping")
                continue

            originals = df[orig_col].astype(str).tolist()
            counterfs = df[cf_col].astype(str).tolist()

            fr, _, _ = flip_rate_multiclass(originals, counterfs, clf_name)
            ss = compute_semantic_similarity(originals, counterfs)
            ppl = compute_perplexity(counterfs)

            summary_rows.append({
                "file": str(file.relative_to(results_dir)),
                "model": model_id,
                "dataset": dataset,
                "setting": "default",
                "flip_rate": round(fr, 3),
                "semantic_similarity": round(ss, 3),
                "perplexity": round(ppl, 2),
            })
            print(f"{dataset} | MODEL {model_id} | FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")

        # ---------------- SNLI ----------------
        elif dataset == "snli":
            prem_col = get_first_existing_col(df, ["premise", "orig_premise"])
            hypo_col = get_first_existing_col(df, ["hypothesis", "orig_hypothesis"])

            if prem_col is None or hypo_col is None:
                print(f"Missing premise/hypothesis columns -> {file.name}, skipping")
                continue

            orig_concat = (
                "Premise: " + df[prem_col].astype(str) +
                " Hypothesis: " + df[hypo_col].astype(str)
            ).tolist()

            
            f = fname
            is_premise_file = "premise" in f and "hypothesis" not in f
            is_hypothesis_file = "hypothesis" in f

            
            did_any = False


            if is_premise_file or (not is_premise_file and not is_hypothesis_file):

                gen_prem_col = get_first_existing_col(
                    df,
                    ["counterfactual_premise", "gen_premise", "cf_premise", "counterfactual_text"]
                )
                if gen_prem_col is not None:
                    counterfs_prem = (
                        "Premise: " + df[gen_prem_col].astype(str) +
                        " Hypothesis: " + df[hypo_col].astype(str)
                    ).tolist()

                    o1, c1 = filter_bad_samples(orig_concat, counterfs_prem)
                    if o1:
                        fr, _, _ = flip_rate_multiclass(o1, c1, clf_name)
                        ss = compute_semantic_similarity(o1, c1)
                        ppl = compute_perplexity(c1)

                        summary_rows.append({
                            "file": str(file.relative_to(results_dir)),
                            "model": model_id,
                            "dataset": dataset,
                            "setting": "premise_replaced",
                            "flip_rate": round(fr, 3),
                            "semantic_similarity": round(ss, 3),
                            "perplexity": round(ppl, 2),
                        })
                        print(f"SNLI-premise_replaced | MODEL {model_id} | FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")
                        did_any = True
                    else:
                        print("No valid SNLI-premise samples after filtering, skipped.")
                else:
                    if is_premise_file:
                        print("No counterfactual_premise-like column found, skipped premise setting.")

            # --- Replace hypothesis ---
            if is_hypothesis_file or (not is_premise_file and not is_hypothesis_file):
                gen_hypo_col = get_first_existing_col(
                    df,
                    ["counterfactual_hypothesis", "gen_hypothesis", "cf_hypothesis", "counterfactual_text"]
                )
                if gen_hypo_col is not None:
                    counterfs_hypo = (
                        "Premise: " + df[prem_col].astype(str) +
                        " Hypothesis: " + df[gen_hypo_col].astype(str)
                    ).tolist()

                    o2, c2 = filter_bad_samples(orig_concat, counterfs_hypo)
                    if o2:
                        fr, _, _ = flip_rate_multiclass(o2, c2, clf_name)
                        ss = compute_semantic_similarity(o2, c2)
                        ppl = compute_perplexity(c2)

                        summary_rows.append({
                            "file": str(file.relative_to(results_dir)),
                            "model": model_id,
                            "dataset": dataset,
                            "setting": "hypothesis_replaced",
                            "flip_rate": round(fr, 3),
                            "semantic_similarity": round(ss, 3),
                            "perplexity": round(ppl, 2),
                        })
                        print(f"SNLI-hypothesis_replaced | MODEL {model_id} | FR {fr:.3f} | SS {ss:.3f} | PPL {ppl:.2f}")
                        did_any = True
                    else:
                        print("No valid SNLI-hypothesis samples after filtering, skipped.")
                else:
                    if is_hypothesis_file:
                        print("No counterfactual_hypothesis-like column found, skipped hypothesis setting.")

            if not did_any:
                print("SNLI: neither premise nor hypothesis setting could be evaluated for this file.")

    except Exception as e:
        print(f"Error -> {file.name} (skipped)")
        print("    Error msg:", e)
        continue


summary_df = pd.DataFrame(summary_rows)

import IPython.display as disp
disp.display(summary_df)

out_path = results_dir / "evaluation_summary.csv"
summary_df.to_csv(out_path, index=False)
print(f"Summary saved to {out_path}")


In [ ]:
import pandas as pd
from pathlib import Path
from iflip.evaluate.metrics import (
    predict_with_sliding_window,
    compute_perplexity,
    compute_semantic_similarity,
)
from iflip.evaluate.metrics import _majority_vote
from transformers import pipeline, AutoTokenizer
from iflip.config import config


MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}


def flip_rate_multiclass(orig_texts, cf_texts):
    clf_name = config.classifier_model
    clf = pipeline("text-classification", model=clf_name, device=0)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)

    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds = predict_with_sliding_window(cf_texts, clf, tok)
    flips = sum(o != c for o, c in zip(orig_preds, cf_preds))
    return flips / len(orig_texts), orig_preds, cf_preds


results_dir = Path("results_causal")
csv_files = sorted(results_dir.glob("*.csv"))
summary_rows = []


for file in csv_files:
    print(f"\n now evaluate: {file.name}")
    try:
        df = pd.read_csv(file)
        df.columns = [c.strip().lower() for c in df.columns]
        orig_col = next(c for c in df.columns if "original" in c)
        cf_col = next(c for c in df.columns if "generated" in c)

        originals = df[orig_col].astype(str).tolist()
        counterfs = df[cf_col].astype(str).tolist()

        dataset = next(ds for ds in MODEL_MAP if ds in file.stem)
        config.task_name = dataset
        config.classifier_model = MODEL_MAP[dataset]

        fr, _, _ = flip_rate_multiclass(originals, counterfs)
        sim = compute_semantic_similarity(originals, counterfs)
        ppl = compute_perplexity(counterfs)

        summary_rows.append(
            {
                "file": file.name,
                "dataset": dataset,
                "flip_rate": round(fr, 3),
                "semantic_similarity": round(sim, 3),
                "perplexity": round(ppl, 2),
            }
        )
        print(f"success | FR {fr:.3f} | SS {sim:.3f} | PPL {ppl:.2f}")

    except Exception as e:
        print(f"Error -> {file.name} (skipped)")
        print(" Error msg:", e)
        continue


pd.DataFrame(summary_rows)